# 05 Anomaly Detection
Compare Z-score, Isolation Forest, and LOF for outlier detection.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Build Feature Matrix

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

feat = ["engagement_rate","view_rate","comment_rate","like_rate","view_engagement_rate","discount_percent","hashtags_count","emoji_count","caption_length"]
X = StandardScaler().fit_transform(df[feat].fillna(0))

## Experiment Grid and Method Comparison

In [4]:
z_vals, conts, neighs = [2,2.5,3], [0.03,0.05,0.1], [10,20,35]
rows, outputs = [], {}

zmax = np.abs(X).max(axis=1)
for z in z_vals:
    y = np.where(zmax >= z, -1, 1)
    out = df.copy()
    out["anomaly_label"] = y
    out["method"] = f"zscore_{z}"
    out["anomaly_type"] = np.where((y==-1) & (out["engagement_rate"] >= out["engagement_rate"].median()), "positive_anomaly", np.where(y==-1, "negative_anomaly", "normal"))
    outputs[f"zscore_{z}"] = out
    rows.append({"method":"zscore","setting":f"threshold={z}","anomalies":int((y==-1).sum())})

for c in conts:
    y = IsolationForest(contamination=c, random_state=42).fit_predict(X)
    out = df.copy()
    out["anomaly_label"] = y
    out["method"] = f"iforest_{c}"
    out["anomaly_type"] = np.where((y==-1) & (out["engagement_rate"] >= out["engagement_rate"].median()), "positive_anomaly", np.where(y==-1, "negative_anomaly", "normal"))
    outputs[f"iforest_{c}"] = out
    rows.append({"method":"isolation_forest","setting":f"contamination={c}","anomalies":int((y==-1).sum())})

for n in neighs:
    y = LocalOutlierFactor(n_neighbors=n, contamination=0.05).fit_predict(X)
    out = df.copy()
    out["anomaly_label"] = y
    out["method"] = f"lof_{n}"
    out["anomaly_type"] = np.where((y==-1) & (out["engagement_rate"] >= out["engagement_rate"].median()), "positive_anomaly", np.where(y==-1, "negative_anomaly", "normal"))
    outputs[f"lof_{n}"] = out
    rows.append({"method":"lof","setting":f"n_neighbors={n}","anomalies":int((y==-1).sum())})

comp = pd.DataFrame(rows)
comp["interpretability"] = comp["method"].map({"zscore":0.95,"isolation_forest":0.75,"lof":0.7})
comp["anomaly_balance"] = 1 - (comp["anomalies"] - comp["anomalies"].median()).abs() / max(comp["anomalies"].max(),1)
ranked = rank_models(comp, higher_is_better_cols=["interpretability","anomaly_balance"])
best = ranked.iloc[0]

overlap_rows = []
keys = list(outputs.keys())
for m1 in keys:
    s1 = set(outputs[m1].index[outputs[m1]["anomaly_label"] == -1].tolist())
    for m2 in keys:
        s2 = set(outputs[m2].index[outputs[m2]["anomaly_label"] == -1].tolist())
        union = max(len(s1 | s2), 1)
        overlap_rows.append({"method_1": m1, "method_2": m2, "jaccard_overlap": len(s1 & s2) / union})
overlap = pd.DataFrame(overlap_rows)

best_key = next(k for k in outputs if str(best["setting"]).split("=")[1] in k and (k.startswith("zscore") if best["method"]=="zscore" else k.startswith("iforest") if best["method"]=="isolation_forest" else k.startswith("lof")))
anomalies = outputs[best_key].query("anomaly_label == -1")[["business_name","sector","post_date","post_type","engagement_rate","view_rate","comment_rate","method","anomaly_type"]]

## Save Outputs

In [5]:
anomalies.to_csv(PROCESSED_DIR / "anomalies.csv", index=False)
ranked.to_csv(REPORTS_DIR / "anomaly_experiments.csv", index=False)
overlap.to_csv(REPORTS_DIR / "anomaly_overlap_matrix.csv", index=False)
display(ranked)
display(anomalies.head(20))
print("Insight: positive anomalies are replication candidates; negative anomalies are intervention candidates.")

,method,setting,anomalies,interpretability,anomaly_balance,composite_score
0,zscore,threshold=3,73,0.95,0.924092,1.909091
1,zscore,threshold=2.5,135,0.95,0.719472,1.664032
2,isolation_forest,contamination=0.05,50,0.75,1.000000,1.200000
3,isolation_forest,contamination=0.03,30,0.75,0.933993,1.120949
4,isolation_forest,contamination=0.1,100,0.75,0.834983,1.002372
5,zscore,threshold=2,303,0.95,0.165017,1.000000
6,lof,n_neighbors=10,50,0.70,1.000000,1.000000
7,lof,n_neighbors=20,50,0.70,1.000000,1.000000
8,lof,n_neighbors=35,50,0.70,1.000000,1.000000


,business_name,sector,post_date,post_type,engagement_rate,view_rate,comment_rate,method,anomaly_type
2,Al Amal Dental,clinic,2025-01-17,carousel,0.055030,0.435727,0.009888,zscore_3,negative_anomaly
27,Al Balad Grill,restaurant,2025-12-04,reel,0.506592,3.621068,0.015664,zscore_3,positive_anomaly
36,Gaza Seaside Kitchen,restaurant,2025-08-15,reel,0.283319,4.948808,0.024272,zscore_3,positive_anomaly
40,Jericho Physio Hub,clinic,2025-12-03,video,0.080776,0.915082,0.007626,zscore_3,negative_anomaly
41,Bethlehem Brew Bar,cafe,2025-12-21,carousel,0.184153,1.334349,0.030475,zscore_3,positive_anomaly
87,Al Amal Dental,clinic,2025-05-08,reel,0.454894,5.551277,0.012128,zscore_3,positive_anomaly
88,Tulkarm Streetwear,store,2025-06-07,image,0.063205,0.682427,0.007627,zscore_3,negative_anomaly
92,Canaan Threads,store,2025-04-23,image,0.175242,0.993015,0.015717,zscore_3,positive_anomaly
117,Jericho Date Bakery,bakery,2025-12-30,reel,0.613111,4.113111,0.026155,zscore_3,positive_anomaly
130,Nablus Sweet Oven,bakery,2025-04-28,reel,0.704013,3.693423,0.061315,zscore_3,positive_anomaly


Insight: positive anomalies are replication candidates; negative anomalies are intervention candidates.
